# 랜덤신호론 14주차 실습
## 랜덤신호 AI 분석모델 통합

1. 지금까지 만든 함수를 하나의 분석 흐름으로 연결한다.
2. 새 신호의 특징을 추출하고 상태를 예측한다.
3. 판단 근거를 특징값과 그래프로 설명한다.
4. 한 가지 조건씩 바꾸어 성능을 비교한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. 기준 모델


In [ ]:
fs=50;df=build_dataset(60,fs);X=df[FEATURE_NAMES];y=df.label
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
model=make_pipeline(StandardScaler(),KNeighborsClassifier(5));model.fit(X_train,y_train)
print('기준 정확도:',accuracy_score(y_test,model.predict(X_test)))


## 2. 통합 분석 함수


In [ ]:
def analyze(signal,fs,model):
    f=extract_features(signal,fs)
    xnew=pd.DataFrame([[f[n] for n in FEATURE_NAMES]],columns=FEATURE_NAMES)
    return model.predict(xnew)[0],f,dict(zip(model.classes_,model.predict_proba(xnew)[0]))
_,new=generate_signal('spike',fs,seed=777);pred,features,prob=analyze(new,fs,model)
print(pred,prob);print(pd.DataFrame([features]))


**관찰 1** 가장 높은 예측 확률과 근거가 될 특징을 찾으세요.


## 3. 시각화와 개선 실험


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4));ax[0].plot(new);ax[0].set_title('Input')
ax[1].bar(list(features),list(features.values()));ax[1].tick_params(axis='x',rotation=70);plt.tight_layout();plt.show()
for k in [3,5,7,11]:
    m=make_pipeline(StandardScaler(),KNeighborsClassifier(k));m.fit(X_train,y_train)
    print(k,accuracy_score(y_test,m.predict(X_test)))


**미니 과제** 기준 모델과 개선 모델의 차이, 한 조건씩 바꾼 이유를 작성하세요.


## 참고 자료
- Pipeline and composite estimators: https://scikit-learn.org/stable/modules/compose.html
- Model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html
